In [5]:
import os
import glob
import json
import time
import warnings
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier, AdaBoostClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
warnings.filterwarnings('ignore')

k_folds = [5, 10]
dataset_ids = list(range(1, 36))
split_ratios = [0.8]
n_iter = 10
inner_splits = 3
n_jobs = -1
print_params = False
top_params = 5

res_dir = '../Results/'
os.makedirs(res_dir, exist_ok=True)

In [6]:
def get_models():
    return {
        'Random Forest': (RandomForestClassifier(random_state=42, n_jobs=1), {
            'n_estimators': [100, 200, 300, 500], 'max_depth': [None, 3, 5, 7, 10],
            'min_samples_split': [2, 5, 10], 'min_samples_leaf': [1, 2, 4], 'max_features': ['sqrt', 'log2', None]}),
        'Logistic Regression': (LogisticRegression(max_iter=3000), {
            'C': [0.001, 0.01, 0.1, 1, 10, 100], 'solver': ['lbfgs', 'liblinear']}),
        'Decision Tree': (DecisionTreeClassifier(random_state=42), {
            'max_depth': [None, 3, 5, 7, 10, 15], 'min_samples_split': [2, 5, 10, 20], 'min_samples_leaf': [1, 2, 4, 8],
            'criterion': ['gini', 'entropy'], 'max_features': [None, 'sqrt', 'log2']}),
        'XGBoost': (XGBClassifier(eval_metric='logloss', random_state=42, n_jobs=1), {
            'n_estimators': [100, 200, 300], 'max_depth': [3, 4, 5, 6, 8], 'learning_rate': [0.01, 0.03, 0.05, 0.1, 0.2],
            'subsample': [0.6, 0.8, 1.0], 'colsample_bytree': [0.6, 0.8, 1.0], 'min_child_weight': [1, 3, 5], 'gamma': [0, 0.1, 0.3]}),
        'CatBoost': (CatBoostClassifier(verbose=0, allow_writing_files=False, random_state=42, thread_count=1), {
            'iterations': [100, 200, 300], 'depth': [3, 4, 6, 8], 'learning_rate': [0.01, 0.03, 0.05, 0.1, 0.2], 'l2_leaf_reg': [1, 3, 5, 7, 9]}),
        'AdaBoost': (AdaBoostClassifier(random_state=42), {
            'n_estimators': [50, 100, 200, 300], 'learning_rate': [0.01, 0.1, 0.5, 1.0, 1.5]}),
        'LightGBM': (LGBMClassifier(verbose=-1, random_state=42, n_jobs=1), {
            'n_estimators': [100, 200, 300], 'num_leaves': [15, 31, 63], 'max_depth': [-1, 3, 5, 7], 'learning_rate': [0.01, 0.03, 0.05, 0.1, 0.2],
            'subsample': [0.6, 0.8, 1.0], 'colsample_bytree': [0.6, 0.8, 1.0], 'min_child_samples': [5, 10, 20], 'reg_lambda': [0, 1, 5]}),
        'Support Vector Machine': (SVC(probability=True, random_state=42), {
            'C': [0.1, 1, 10, 100], 'kernel': ['rbf', 'linear'], 'gamma': ['scale', 'auto', 0.01, 0.1, 1]}),
        'K-Nearest Neighbors': (KNeighborsClassifier(), {
            'n_neighbors': [3, 5, 7, 9, 11, 15, 21], 'weights': ['uniform', 'distance'], 'p': [1, 2]}),
        'Extra Trees': (ExtraTreesClassifier(random_state=42, n_jobs=1), {
            'n_estimators': [100, 200, 300, 500], 'max_depth': [None, 3, 5, 7, 10],
            'min_samples_split': [2, 5, 10], 'min_samples_leaf': [1, 2, 4], 'max_features': ['sqrt', 'log2', None]}),
        'Gradient Boosting': (GradientBoostingClassifier(random_state=42), {
            'n_estimators': [50, 100, 200, 300], 'learning_rate': [0.01, 0.05, 0.1, 0.2], 'max_depth': [2, 3, 4, 5],
            'subsample': [0.6, 0.8, 1.0], 'min_samples_split': [2, 5, 10], 'min_samples_leaf': [1, 2, 4]}),
        'Naive Bayes': (GaussianNB(), {
            'var_smoothing': [1e-9, 1e-8, 1e-7, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1]})
    }

def grid_size(grid):
    size = 1
    for values in grid.values():
        size *= len(values)
    return size

def to_py(value):
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return float(value)
    return value

In [7]:
def run_cv(X, y, key, dataset_number):
    fold_rows = []
    param_rows = []
    candidate_rows = []
    inner_cv = StratifiedKFold(n_splits=inner_splits, shuffle=True, random_state=42)
    for k in k_folds:
        skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
        for fold, (train_index, test_index) in enumerate(skf.split(X, y), start=1):
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train, y_test = y.iloc[train_index], y.iloc[test_index]
            tuned = {}
            probas = {}
            preds = {}
            inner_scores = {}
            train_times = {}
            for name, (estimator, grid) in get_models().items():
                search = RandomizedSearchCV(estimator, grid, n_iter=min(n_iter, grid_size(grid)), cv=inner_cv, scoring='roc_auc', random_state=42, n_jobs=n_jobs)
                start_time = time.time()
                search.fit(X_train, y_train)
                train_time = time.time() - start_time
                best = search.best_estimator_
                start_time = time.time()
                proba = best.predict_proba(X_test)[:, 1]
                pred = best.predict(X_test)
                test_time = time.time() - start_time
                accuracy = accuracy_score(y_test, pred)
                auc_value = roc_auc_score(y_test, proba)
                params = {p: to_py(v) for p, v in search.best_params_.items()}
                tuned[name] = best
                probas[name] = proba
                preds[name] = pred
                inner_scores[name] = search.best_score_
                train_times[name] = train_time
                fold_rows.append({'Ratio': key, 'Dataset': 'Dataset_' + str(dataset_number), 'No': dataset_number, 'K-Fold': k, 'Fold': fold, 'Classifier': name,
                                  'Accuracy': accuracy, 'AUC': auc_value, 'Training Time (s)': train_time, 'Testing Time (s)': test_time})
                param_rows.append({'Ratio': key, 'Dataset': 'Dataset_' + str(dataset_number), 'No': dataset_number, 'K-Fold': k, 'Fold': fold, 'Classifier': name,
                                   'Inner AUC': search.best_score_, 'Test AUC': auc_value, 'Best Params': json.dumps(params, sort_keys=True)})
                if print_params:
                    cv_table = pd.DataFrame(search.cv_results_)[['params', 'mean_test_score', 'rank_test_score']].sort_values('mean_test_score', ascending=False).head(top_params)
                    print(key, 'Dataset', dataset_number, 'K', k, 'Fold', fold, name)
                    print(cv_table.to_string(index=False))
                    for _, r in cv_table.iterrows():
                        candidate_rows.append({'Ratio': key, 'Dataset': 'Dataset_' + str(dataset_number), 'No': dataset_number, 'K-Fold': k, 'Fold': fold, 'Classifier': name,
                                               'AUC': r['mean_test_score'], 'Rank': r['rank_test_score'], 'Params': str(r['params'])})
            ranking = sorted(tuned.keys(), key=lambda n: inner_scores[n], reverse=True)
            for top in [3, 5, 7]:
                selected = ranking[:top]
                estimators = [(n, clone(tuned[n])) for n in selected]
                stack = StackingClassifier(estimators=estimators, final_estimator=LogisticRegression(max_iter=3000), cv=5, n_jobs=n_jobs)
                start_time = time.time()
                stack.fit(X_train, y_train)
                train_time = time.time() - start_time
                start_time = time.time()
                proba = stack.predict_proba(X_test)[:, 1]
                pred = stack.predict(X_test)
                test_time = time.time() - start_time
                fold_rows.append({'Ratio': key, 'Dataset': 'Dataset_' + str(dataset_number), 'No': dataset_number, 'K-Fold': k, 'Fold': fold, 'Classifier': 'Stacking Top ' + str(top),
                                  'Accuracy': accuracy_score(y_test, pred), 'AUC': roc_auc_score(y_test, proba), 'Training Time (s)': train_time, 'Testing Time (s)': test_time})
                base_train = sum(train_times[n] for n in selected)
                start_time = time.time()
                prob_matrix = np.column_stack([probas[n] for n in selected])
                pred_matrix = np.column_stack([preds[n] for n in selected])
                weights = np.array([inner_scores[n] for n in selected])
                votes = {
                    'Hard Voting': pred_matrix.mean(axis=1),
                    'Soft Voting': prob_matrix.mean(axis=1),
                    'Weighted Voting': np.average(prob_matrix, axis=1, weights=weights)
                }
                vote_time = time.time() - start_time
                for vote_name, score in votes.items():
                    fold_rows.append({'Ratio': key, 'Dataset': 'Dataset_' + str(dataset_number), 'No': dataset_number, 'K-Fold': k, 'Fold': fold, 'Classifier': vote_name + ' Top ' + str(top),
                                      'Accuracy': accuracy_score(y_test, (score >= 0.5).astype(int)), 'AUC': roc_auc_score(y_test, score),
                                      'Training Time (s)': base_train, 'Testing Time (s)': vote_time})
    return pd.DataFrame(fold_rows), pd.DataFrame(param_rows), pd.DataFrame(candidate_rows)

In [8]:
for ratio in split_ratios:
    for dataset_number in dataset_ids:
        key = str(ratio)
        tag = key + '_Dataset_' + str(dataset_number)
        if os.path.exists(res_dir + 'CV_Fold_Results_' + tag + '.pkl'):
            continue
        folder = '../Datasets/Split_' + str(round(ratio * 100)) + '/'
        data = pd.read_csv(folder + 'Dataset_' + str(dataset_number) + '_train.csv')
        X = data.drop('label', axis=1)
        y = data['label']
        start_time = time.time()
        fold_df, param_df, candidate_df = run_cv(X, y, key, dataset_number)
        fold_df.to_pickle(res_dir + 'CV_Fold_Results_' + tag + '.pkl')
        param_df.to_pickle(res_dir + 'CV_Params_' + tag + '.pkl')
        candidate_df.to_pickle(res_dir + 'CV_Candidates_' + tag + '.pkl')
        print('Ratio', ratio, 'Dataset', dataset_number, 'finished in', round((time.time() - start_time) / 60, 2), 'minutes')

Ratio 0.8 Dataset 1 finished in 4.0 minutes
Ratio 0.8 Dataset 2 finished in 2.31 minutes
Ratio 0.8 Dataset 3 finished in 2.72 minutes
Ratio 0.8 Dataset 4 finished in 3.26 minutes
Ratio 0.8 Dataset 5 finished in 3.14 minutes
Ratio 0.8 Dataset 6 finished in 3.24 minutes
Ratio 0.8 Dataset 7 finished in 3.0 minutes
Ratio 0.8 Dataset 8 finished in 3.39 minutes
Ratio 0.8 Dataset 9 finished in 3.26 minutes
Ratio 0.8 Dataset 10 finished in 2.99 minutes
Ratio 0.8 Dataset 11 finished in 3.17 minutes
Ratio 0.8 Dataset 12 finished in 4.3 minutes
Ratio 0.8 Dataset 13 finished in 3.49 minutes
Ratio 0.8 Dataset 14 finished in 4.74 minutes
Ratio 0.8 Dataset 15 finished in 4.09 minutes
Ratio 0.8 Dataset 16 finished in 3.54 minutes
Ratio 0.8 Dataset 17 finished in 3.66 minutes
Ratio 0.8 Dataset 18 finished in 8.41 minutes
Ratio 0.8 Dataset 19 finished in 13.12 minutes
Ratio 0.8 Dataset 20 finished in 4.77 minutes
Ratio 0.8 Dataset 21 finished in 31.67 minutes
Ratio 0.8 Dataset 22 finished in 3.63 minute

In [9]:
results_df = pd.concat([pd.read_pickle(f) for f in sorted(glob.glob(res_dir + 'CV_Fold_Results_*.pkl'))], ignore_index=True)
summary_df = results_df.groupby(['Ratio', 'No', 'Dataset', 'K-Fold', 'Classifier']).agg(Accuracy=('Accuracy', 'mean'), AUC=('AUC', 'mean'), Training=('Training Time (s)', 'mean'), Testing=('Testing Time (s)', 'mean')).reset_index()
summary_df.to_csv(res_dir + 'CV_Summary.csv', index=False)
summary_df

,Ratio,No,Dataset,K-Fold,Classifier,Accuracy,AUC,Training,Testing
0,0.7,1,Dataset_1,5,AdaBoost,0.744592,0.806307,1.479893,0.046460
1,0.7,1,Dataset_1,5,CatBoost,0.748936,0.823813,2.580424,0.005090
2,0.7,1,Dataset_1,5,Decision Tree,0.748848,0.755261,0.152425,0.002257
3,0.7,1,Dataset_1,5,Extra Trees,0.736170,0.816405,1.938073,0.036841
4,0.7,1,Dataset_1,5,Gradient Boosting,0.744770,0.812527,1.467033,0.003149
...,...,...,...,...,...,...,...,...,...
2155,0.8,35,Dataset_35,10,Support Vector Machine,0.707407,0.780380,0.156732,0.000101
2156,0.8,35,Dataset_35,10,Weighted Voting Top 3,0.728439,0.813773,1.818656,0.000000
2157,0.8,35,Dataset_35,10,Weighted Voting Top 5,0.714153,0.810585,4.493216,0.000000
2158,0.8,35,Dataset_35,10,Weighted Voting Top 7,0.721561,0.805003,6.512711,0.000000


In [10]:
pivot_df1 = summary_df.pivot_table(index=['Ratio', 'No', 'K-Fold'], columns='Classifier', values='Accuracy')
pivot_df1

Classifier       AdaBoost  CatBoost  Decision Tree  Extra Trees  \
Ratio No K-Fold                                                   
0.7   1  5       0.744592  0.748936       0.748848     0.736170   
         10      0.740399  0.748913       0.736051     0.677717   
         15      0.761111  0.723611       0.701944     0.701944   
         20      0.752652  0.703030       0.701894     0.735985   
         30      0.748810  0.745238       0.672024     0.731548   
...                   ...       ...            ...          ...   
0.8   33 10      0.721693  0.728968       0.721825     0.695767   
      34 5       0.765657  0.732795       0.725657     0.681818   
         10      0.769180  0.754894       0.725529     0.688624   
      35 5       0.743838  0.736431       0.684916     0.670303   
         10      0.732540  0.717989       0.692460     0.666931   

Classifier       Gradient Boosting  Hard Voting Top 3  Hard Voting Top 5  \
Ratio No K-Fold                                                            
0.7   1  5                0.744770           0.744858           0.753280   
         10               0.736051           0.736232           0.736413   
         15               0.748889           0.740278           0.731944   
         20               0.723864           0.740530           0.715530   
         30               0.749405           0.744643           0.757738   
...                            ...                ...                ...   
0.8   33 10               0.736640           0.725397           0.736376   
      34 5                0.717710           0.769293           0.754680   
         10               0.736772           0.747487           0.762169   
      35 5                0.699731           0.729024           0.714411   
         10               0.674074           0.706481           0.710582   

Classifier       Hard Voting Top 7  K-Nearest Neighbors  LightGBM  ...  \
Ratio No K-Fold                                                    ...   
0.7   1  5                0.761525             0.719858  0.732181  ...   
         10               0.744746             0.711775  0.736413  ...   
         15               0.736111             0.707222  0.719167  ...   
         20               0.723864             0.711742  0.728030  ...   
         30               0.740476             0.695238  0.739881  ...   
...                            ...                  ...       ...  ...   
0.8   33 10               0.739947             0.703571  0.728968  ...   
      34 5                0.747542             0.714613  0.747340  ...   
         10               0.751323             0.696296  0.769444  ...   
      35 5                0.725387             0.670370  0.714478  ...   
         10               0.706878             0.670635  0.692063  ...   

Classifier       Soft Voting Top 5  Soft Voting Top 7  Stacking Top 3  \
Ratio No K-Fold                                                         
0.7   1  5                0.757535           0.770035        0.753191   
         10               0.732609           0.728080        0.744746   
         15               0.736389           0.723611        0.749167   
         20               0.719697           0.736364        0.728030   
         30               0.757738           0.741071        0.752976   
...                            ...                ...             ...   
0.8   33 10               0.721693           0.725397        0.714550   
      34 5                0.750976           0.740000        0.740000   
         10               0.766005           0.751323        0.758598   
      35 5                0.725455           0.725522        0.736296   
         10               0.714153           0.721561        0.714153   

Classifier       Stacking Top 5  Stacking Top 7  Support Vector Machine  \
Ratio No K-Fold                                                           
0.7   1  5             0.765869        0.774202                0.699025   
 

In [11]:
pivot_df2 = summary_df.pivot_table(index=['Ratio', 'No', 'K-Fold'], columns='Classifier', values='AUC')
pivot_df2

Classifier       AdaBoost  CatBoost  Decision Tree  Extra Trees  \
Ratio No K-Fold                                                   
0.7   1  5       0.806307  0.823813       0.755261     0.816405   
         10      0.813333  0.837870       0.763194     0.792500   
         15      0.844444  0.825111       0.735222     0.797667   
         20      0.833214  0.813482       0.752567     0.812232   
         30      0.842222  0.836667       0.715556     0.817778   
...                   ...       ...            ...          ...   
0.8   33 10      0.803886  0.820710       0.761768     0.802466   
      34 5       0.806409  0.809048       0.788297     0.778961   
         10      0.844872  0.832344       0.777011     0.778461   
      35 5       0.821643  0.817859       0.795703     0.778406   
         10      0.819020  0.810517       0.787414     0.779150   

Classifier       Gradient Boosting  Hard Voting Top 3  Hard Voting Top 5  \
Ratio No K-Fold                                                            
0.7   1  5                0.812527           0.776656           0.793649   
         10               0.824074           0.758333           0.771250   
         15               0.848667           0.759444           0.784000   
         20               0.812054           0.751473           0.763504   
         30               0.821111           0.769444           0.796111   
...                            ...                ...                ...   
0.8   33 10               0.798601           0.732059           0.767157   
      34 5                0.797008           0.763112           0.760783   
         10               0.804608           0.773500           0.782274   
      35 5                0.816108           0.741548           0.766279   
         10               0.819060           0.729070           0.752059   

Classifier       Hard Voting Top 7  K-Nearest Neighbors  LightGBM  ...  \
Ratio No K-Fold                                                    ...   
0.7   1  5                0.793747             0.723747  0.821852  ...   
         10               0.771667             0.740185  0.807500  ...   
         15               0.793667             0.746556  0.818222  ...   
         20               0.786629             0.727879  0.811964  ...   
         30               0.815000             0.753333  0.816667  ...   
...                            ...                  ...       ...  ...   
0.8   33 10               0.774658             0.697184  0.812460  ...   
      34 5                0.775500             0.755472  0.787387  ...   
         10               0.795511             0.763617  0.813999  ...   
      35 5                0.778223             0.704529  0.812979  ...   
         10               0.764173             0.707732  0.807497  ...   

Classifier       Soft Voting Top 5  Soft Voting Top 7  Stacking Top 3  \
Ratio No K-Fold                                                         
0.7   1  5                0.839390           0.835621        0.834662   
         10               0.832593           0.840463        0.820556   
         15               0.851778           0.850667        0.842222   
         20               0.827098           0.833304        0.815223   
         30               0.847778           0.856667        0.872222   
...                            ...                ...             ...   
0.8   33 10               0.827641           0.828271        0.822683   
      34 5                0.811261           0.801992        0.808773   
         10               0.834421           0.832389        0.836916   
      35 5                0.819466           0.817141        0.821066   
         10               0.809516           0.803827        0.809688   

Classifier       Stacking Top 5  Stacking Top 7  Support Vector Machine  \
Ratio No K-Fold                                                           
0.7   1  5             0.844967        0.831961                0.767930   
 

In [12]:
pivot_df3 = summary_df.pivot_table(index=['Ratio', 'No', 'K-Fold'], columns='Classifier', values=['Training', 'Testing'])
pivot_df1.to_csv(res_dir + 'CV_Accuracy.csv')
pivot_df2.to_csv(res_dir + 'CV_AUC.csv')
pivot_df3.to_csv(res_dir + 'CV_Times.csv')
pivot_df3

Testing                                      \
Classifier       AdaBoost  CatBoost Decision Tree Extra Trees   
Ratio No K-Fold                                                 
0.7   1  5       0.046460  0.005090      0.002257    0.036841   
         10      0.047329  0.006241      0.001732    0.041244   
         15      0.044288  0.001096      0.000709    0.023218   
         20      0.046771  0.005439      0.001812    0.041744   
         30      0.046679  0.004840      0.002446    0.030721   
...                   ...       ...           ...         ...   
0.8   33 10      0.013971  0.001452      0.000611    0.011264   
      34 5       0.024298  0.003055      0.000000    0.011810   
         10      0.017248  0.003045      0.000813    0.015373   
      35 5       0.018319  0.002767      0.001625    0.016726   
         10      0.018281  0.000189      0.000906    0.013576   

                                                                       \
Classifier      Gradient Boosting Hard Voting Top 3 Hard Voting Top 5   
Ratio No K-Fold                                                         
0.7   1  5               0.003149          0.000000          0.000000   
         10              0.001271          0.000000          0.000200   
         15              0.003094          0.000067          0.000000   
         20              0.001675          0.000101          0.000105   
         30              0.003716          0.000146          0.000155   
...                           ...               ...               ...   
0.8   33 10              0.000608          0.000000          0.000000   
      34 5               0.002771          0.000000          0.000000   
         10              0.004233          0.000000          0.000000   
      35 5               0.003368          0.000200          0.000000   
         10              0.002631          0.000000          0.000000   

                                                                 ...  \
Classifier      Hard Voting Top 7 K-Nearest Neighbors  LightGBM  ...   
Ratio No K-Fold                                                  ...   
0.7   1  5               0.000000            0.030043  0.005021  ...   
         10              0.000000            0.027015  0.004332  ...   
         15              0.000000            0.016536  0.000199  ...   
         20              0.000142            0.025221  0.003409  ...   
         30              0.000147            0.024495  0.003274  ...   
...                           ...                 ...       ...  ...   
0.8   33 10              0.000000            0.012434  0.003006  ...   
      34 5               0.000000            0.000633  0.000000  ...   
         10              0.000100            0.000992  0.000377  ...   
      35 5               0.000000            0.000401  0.000000  ...   
         10              0.000000            0.000850  0.000681  ...   

                         Training                                   \
Classifier      Soft Voting Top 5 Soft Voting Top 7 Stacking Top 3   
Ratio No K-Fold                                                      
0.7   1  5               8.431597         11.059166       3.326611   
         10              7.315952         10.302575       2.900841   
         15              7.099540          8.942390       2.937024   
         20              8.557114         11.260701       3.556513   
         30              8.001439         10.011136       3.514147   
...                           ...               ...            ...   
0.8   33 10              4.889739          6.963587       1.990935   
      34 5               3.654647          5.001176       1.452989   
         10              3.789288          5.255189       1.195052   
      35 5               4.588765          6.586646       1.859968   
         10              4.493216          6.512711       0.605241   

                                                                      \
Classifier      Stacki

In [13]:
param_all = pd.concat([pd.read_pickle(f) for f in sorted(glob.glob(res_dir + 'CV_Params_*.pkl'))], ignore_index=True)
candidate_files = sorted(glob.glob(res_dir + 'CV_Candidates_*.pkl'))
candidate_all = pd.concat([pd.read_pickle(f) for f in candidate_files], ignore_index=True) if len(candidate_files) > 0 else pd.DataFrame()
show_dataset = 1
show_classifier = 'Random Forest'
show_rows = 20
param_view = param_all[(param_all['No'] == show_dataset) & (param_all['Classifier'] == show_classifier)].sort_values('Inner AUC', ascending=False)
param_view[['Ratio', 'K-Fold', 'Fold', 'Inner AUC', 'Test AUC', 'Best Params']].head(show_rows)

,Ratio,K-Fold,Fold,Inner AUC,Test AUC,Best Params
3948,0.8,10,5,0.880158,0.729412,"{""max_depth"": 5, ""max_features"": ""sqrt"", ""min_..."
600,0.7,30,1,0.874734,0.333333,"{""max_depth"": 10, ""max_features"": ""log2"", ""min..."
300,0.7,15,11,0.871644,0.733333,"{""max_depth"": 5, ""max_features"": ""sqrt"", ""min_..."
708,0.7,30,10,0.862009,0.933333,"{""max_depth"": 7, ""max_features"": ""sqrt"", ""min_..."
528,0.7,20,15,0.861871,0.800000,"{""max_depth"": 7, ""max_features"": ""log2"", ""min_..."
0,0.7,5,1,0.860613,0.757407,"{""max_depth"": 7, ""max_features"": ""log2"", ""min_..."
3984,0.8,10,8,0.860134,0.852941,"{""max_depth"": 7, ""max_features"": ""log2"", ""min_..."
840,0.7,30,21,0.859651,0.466667,"{""max_depth"": 7, ""max_features"": ""sqrt"", ""min_..."
3900,0.8,10,1,0.857226,0.772222,"{""max_depth"": 7, ""max_features"": ""sqrt"", ""min_..."
60,0.7,10,1,0.856664,0.829630,"{""max_depth"": 7, ""max_features"": ""sqrt"", ""min_..."


In [14]:
if len(candidate_all) > 0:
    candidate_view = candidate_all[(candidate_all['No'] == show_dataset) & (candidate_all['Classifier'] == show_classifier)].sort_values('AUC', ascending=False)
    print(candidate_view[['Ratio', 'K-Fold', 'Fold', 'AUC', 'Rank', 'Params']].head(show_rows).to_string(index=False))

In [15]:
selection_mode = 'mode'
best_params = {}
for (ratio, number, name), g in param_all.groupby(['Ratio', 'No', 'Classifier']):
    counts = g.groupby('Best Params').agg(count=('Inner AUC', 'size'), inner=('Inner AUC', 'mean')).reset_index()
    if selection_mode == 'mode':
        chosen = counts.sort_values(['count', 'inner'], ascending=False).iloc[0]
    else:
        chosen = counts.sort_values(['inner', 'count'], ascending=False).iloc[0]
    best_params.setdefault(str(ratio), {}).setdefault('Dataset_' + str(number), {})[name] = json.loads(chosen['Best Params'])
json.dump(best_params, open(res_dir + 'best_params.json', 'w'), indent=2)
len(best_params)

2

In [20]:
summary_df.sort_values('AUC', ascending=False).head(20)

,Ratio,No,Dataset,K-Fold,Classifier,Accuracy,AUC,Training,Testing
1624,0.8,24,Dataset_24,10,Stacking Top 3,0.772354,0.875253,1.696605,0.024176
112,0.7,1,Dataset_1,30,Stacking Top 3,0.752976,0.872222,3.514147,0.087045
1622,0.8,24,Dataset_24,10,Soft Voting Top 5,0.772619,0.870633,5.287558,0.000000
1629,0.8,24,Dataset_24,10,Weighted Voting Top 5,0.772619,0.870633,5.287558,0.000000
1625,0.8,24,Dataset_24,10,Stacking Top 5,0.768915,0.870098,2.506226,0.047979
1628,0.8,24,Dataset_24,10,Weighted Voting Top 3,0.779762,0.869575,3.054707,0.000000
1621,0.8,24,Dataset_24,10,Soft Voting Top 3,0.779762,0.869575,3.054707,0.000000
1704,0.8,26,Dataset_26,10,AdaBoost,0.784127,0.868619,2.418354,0.049112
1631,0.8,24,Dataset_24,10,XGBoost,0.776190,0.867956,0.268408,0.005119
1512,0.8,22,Dataset_22,10,AdaBoost,0.783862,0.867692,1.265750,0.016667
